## Лабораторная работа № 1
## Выполнение разведочного анализа больших данных с использованием фреймворка Apache Spark

### Часть 2. Разведочный анализ (EDA)

В этой части мы по каждому признаку датасета:
* определим **тип** (количественный / категориальный / бинарный);
* найдём **пропущенные значения** и устраним их;
* найдём **выбросы** и устраним их;
* рассчитаем **статистические показатели** (средние, квартили);
* построим **распределения** важнейших признаков;
* рассчитаем **корреляции** между количественными признаками.

Итог — очищенная таблица `lab1_flights_processed_table`, на которой в лабораторной работе № 2
будут обучаться модели машинного обучения.

### Подключение библиотек

К знакомым по части 1 Spark-классам добавляются:
* `pandas`, `seaborn`, `matplotlib` — графики строим на драйвере: Spark считает агрегаты по всем данным,
  а на визуализацию отправляем уже компактную сводку (сэмпл);
* `F` (`pyspark.sql.functions`) — колонки, агрегаты, приведение типов.

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, DataFrame
from pyspark import SparkConf
from pyspark.sql import functions as F

### Конфигурация и запуск Spark-сессии

Конфигурация полностью повторяет часть 1: YARN, client-режим, Iceberg-каталог в HDFS.

In [ ]:
def create_spark_configuration() -> SparkConf:
    """Создаёт и настраивает конфигурацию Spark-сессии для работы на кластере."""
    user_name = os.getenv("USER")

    conf = SparkConf()
    conf.setAppName("SOBD lab1 — flights EDA")
    conf.setMaster("yarn")
    conf.set("spark.submit.deployMode", "client")
    conf.set("spark.executor.memory", "12g")
    conf.set("spark.executor.cores", "8")
    conf.set("spark.executor.instances", "2")
    conf.set("spark.driver.memory", "4g")
    conf.set("spark.driver.cores", "2")
    conf.set("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0")
    conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    conf.set("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog")
    conf.set("spark.sql.catalog.spark_catalog.type", "hadoop")
    conf.set("spark.sql.catalog.spark_catalog.warehouse", f"hdfs:///user/{user_name}/warehouse")
    conf.set("spark.sql.catalog.spark_catalog.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
    return conf

spark = SparkSession.builder.config(conf=create_spark_configuration()).getOrCreate()
spark

### Загрузка таблицы из части 1

Читаем таблицу `lab1_flights_table` из Iceberg. В отличие от CSV, здесь уже известна схема —
Spark читает Parquet-файлы и метаданные таблицы, не сканируя данные.

In [ ]:
spark.catalog.setCurrentDatabase("sidskii_database")
df = spark.table("lab1_flights_table")
df.show(5)

In [ ]:
print(f"Строк в датасете: {df.count():,}")
df.printSchema()

### Вспомогательные функции

Три универсальных инструмента анализа (по образцу методички):
* `count_nulls` — считает null/not-null в столбце;
* `plot_cat_distribution` — распределение категориального признака (топ-N категорий);
* `plot_quant_distribution` — гистограмма количественного признака: Spark считает частоты
  по бакетам распределённо, на драйвер уходит только ~100 чисел.

In [ ]:
def count_nulls(data: DataFrame, column_name: str) -> None:
    """Печатает количество null и not null значений в столбце."""
    total = data.count()
    nulls = data.filter(F.col(column_name).isNull()).count()
    print(f"{column_name}: null = {nulls:,} ({nulls / total:.1%}), not null = {total - nulls:,}")


def plot_cat_distribution(data: DataFrame, column_name: str, top_n: int = 20) -> None:
    """Столбчатая диаграмма топ-N категорий по частоте."""
    counts = (
        data.groupBy(column_name).count().orderBy(F.desc("count"))
        .limit(top_n).toPandas()  # агрегат уже маленький — забираем на драйвер
    )
    plt.figure(figsize=(12, 4))
    sns.barplot(data=counts, x=column_name, y="count", color="steelblue")
    plt.title(f"Распределение признака {column_name} (топ-{top_n})")
    plt.xticks(rotation=60)
    plt.tight_layout()
    plt.show()


def plot_quant_distribution(data: DataFrame, column: str, num_bins: int = 100,
                            lo: float = None, hi: float = None) -> None:
    """Гистограмма количественного признака: бакеты считаются в Spark."""
    col = F.col(column).cast("double")
    lo = data.select(F.min(col)).first()[0] if lo is None else lo
    hi = data.select(F.max(col)).first()[0] if hi is None else hi
    step = (hi - lo) / num_bins
    hist = (
        data.select(F.floor((col - lo) / step).cast("int").alias("bucket"))
        .groupBy("bucket").count().orderBy("bucket").toPandas()
    )
    hist["x"] = lo + hist["bucket"] * step
    plt.figure(figsize=(12, 3))
    sns.barplot(data=hist, x="x", y="count", color="steelblue")
    plt.title(f"Распределение признака {column}")
    step_t = max(1, num_bins // 15)
    plt.xticks(range(0, num_bins, step_t), [f"{v:.0f}" for v in hist['x'][::step_t]], rotation=45)
    plt.tight_layout()
    plt.show()

### Анализ статусов рейса: `Cancelled`, `Diverted`, `CancellationCode`

Начнём с особой группы признаков — статусов рейса. Интуиция: у **отменённых** и **посаженных в
другой аэропорт** рейсов целевой признак `ArrDelay` не определён (рейс не состоялся).
Проверим и посмотрим на пропорции.

In [ ]:
df.groupBy("Cancelled").count().show()
df.groupBy("Diverted").count().show()

In [ ]:
# Код причины отмены: A — авиакомпания, B — погода, C — УВД, D — безопасность
df.filter(F.col("Cancelled") == 1).groupBy("CancellationCode").count().show()

Отменённые и перенаправленные рейсы составляют малую долю данных (~2–3%), но для них
не определены все «фактические» признаки (`ArrDelay`, `AirTime`, `TaxiIn/Out`, `DepTime`...).

**Решение:** удаляем такие рейсы — для задач прогнозирования задержки это некорректные объекты
(в момент прогноза мы всегда предполагаем, что рейс состоялся). Вместе с ними теряет смысл
и столбец `CancellationCode` (после фильтрации он пуст на 100%) — удаляем и его.

In [ ]:
df = df.filter((F.col("Cancelled") == 0) & (F.col("Diverted") == 0)) \
       .drop("Cancelled", "Diverted", "CancellationCode")

print(f"Строк после удаления отменённых и перенаправленных: {df.count():,}")

### Целевой признак `ArrDelay` (задержка прибытия, мин)

Это наша будущая цель регрессии в лабе 2. Признак **количественный**.
Сначала — статистики и квартили, затем распределение и выбросы.

In [ ]:
count_nulls(df, "ArrDelay")
df.select("ArrDelay").summary("count", "min", "25%", "50%", "75%", "max", "mean").show()

In [ ]:
plot_quant_distribution(df, "ArrDelay", num_bins=120, lo=-60, hi=400)

Картина типична для задержек: острая вершина около нуля (большинство рейсов вовремя или чуть раньше),
длинный «хвост» вправо — редкие задержки в сотни минут. Отрицательные значения — **ранние прилёты**,
это не ошибка данных.

**Выбросы.** Хвост тянет среднее и портит регрессию. Посмотрим на долю экстремальных значений
и обрежем диапазон разумными границами: **[-60; 300] минут** (5 часов — уже аномалия,
но ещё реальная практика аэропортов).

In [ ]:
outliers = df.filter((F.col("ArrDelay") > 300) | (F.col("ArrDelay") < -60)).count()
print(f"Выбросов за границами [-60; 300]: {outliers:,} ({outliers / df.count():.2%})")

df = df.filter((F.col("ArrDelay") >= -60) & (F.col("ArrDelay") <= 300))
plot_quant_distribution(df, "ArrDelay", num_bins=120, lo=-60, hi=300)

### Признак `DepDelay` (задержка вылета, мин)

Количественный, ближайший «родственник» целевого. Обрабатываем аналогично.

In [ ]:
count_nulls(df, "DepDelay")
df.select("DepDelay").summary("min", "50%", "max").show()

df = df.filter((F.col("DepDelay") >= -50) & (F.col("DepDelay") <= 300))
plot_quant_distribution(df, "DepDelay", num_bins=100, lo=-50, hi=300)

### Причины задержки: `CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay`

По стандарту BTS эти столбцы заполняются **только для задержанных рейсов** — для всех остальных
там null, который по смыслу означает «0 минут по этой причине». Проверим и заполним нулями.

In [ ]:
delay_causes = ["CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"]
for c in delay_causes:
    count_nulls(df, c)

In [ ]:
df = df.fillna({c: 0 for c in delay_causes})
count_nulls(df, "CarrierDelay")

### Анализ категориальных признаков

`Reporting_Airline` — авиакомпания; `Origin`/`Dest` — аэропорты (тысячи категорий!);
`OriginCityName`/`DestCityName` — города. Для высококардинальных признаков смотрим топ-20.

In [ ]:
count_nulls(df, "Reporting_Airline")
plot_cat_distribution(df, "Reporting_Airline")

In [ ]:
plot_cat_distribution(df, "Origin", top_n=20)

In [ ]:
plot_cat_distribution(df, "Dest", top_n=20)

Топ аэропортов ожидаемо возглавляют Atlanta, Dallas/Fort Worth, Denver — крупнейшие хабы США.
Пропусков в категориальных признаках нет, кодирование отложим до лабы 2 (там этим займётся
`StringIndexer` в ML-конвейере).

### Анализ календарных признаков: `FlightDate`, `DayOfWeek`

Из даты извлечём **месяц** и **год** — они пригодятся для сезонного анализа и как признаки модели.
`DayOfWeek` (1 — понедельник ... 7 — воскресенье) посмотрим на распределение.

In [ ]:
df = df.withColumn("Month", F.month("FlightDate"))
plot_cat_distribution(df, "Month")
plot_cat_distribution(df, "DayOfWeek")

### Анализ времени вылета: `CRSDepTime`

Плановое время в формате ЧЧММ (например, 635 = 06:35). Как число оно почти бесполезно для модели
(2355 вдруг «больше» 005), поэтому извлечём **час вылета** — классический признак,
влияющий на задержки (утренние рейсы задерживаются реже).

In [ ]:
df = df.withColumn("DepHour", F.floor(F.col("CRSDepTime") / 100).cast("int"))
plot_cat_distribution(df, "DepHour")

### Анализ количественных признаков: `Distance`, `AirTime`, `TaxiIn/Out`, `CRSElapsedTime`

Проверим пропуски и статистики. `Distance` — дальность маршрута в милях; хвост до ~5000 миль —
это реальные дальнемагистральные рейсы (например, Гуам–Гонолулу), **не выбросы**, оставляем.

In [ ]:
for c in ["Distance", "AirTime", "TaxiIn", "TaxiOut", "CRSElapsedTime", "ActualElapsedTime"]:
    count_nulls(df, c)

In [ ]:
df.select("Distance", "AirTime", "TaxiIn", "TaxiOut").summary(
    "min", "25%", "50%", "75%", "max", "mean"
).show()

In [ ]:
plot_quant_distribution(df, "Distance", num_bins=100, lo=0, hi=3000)
plot_quant_distribution(df, "TaxiOut", num_bins=100, lo=0, hi=100)

Одиночные null в этих столбцах после фильтрации остались (доли процента) — удалим такие строки
целиком: потери микроскопические, а данные будут полностью чистыми.

In [ ]:
before = df.count()
df = df.dropna(subset=["Distance", "AirTime", "TaxiIn", "TaxiOut",
                       "CRSElapsedTime", "ActualElapsedTime", "DepTime", "ArrTime"])
print(f"Удалено строк с редкими пропусками: {before - df.count():,}")

### Средняя задержка по категориям — ищем закономерности

EDA — это не только про чистоту данных, но и про инсайты. Посмотрим, как средняя задержка
прибытия зависит от часа вылета и дня недели. Каждая агрегация — распределённый запрос по 31 млн строк.

In [ ]:
by_hour = df.groupBy("DepHour").agg(F.mean("ArrDelay").alias("avg_delay"),
                                     F.count("*").alias("flights")).toPandas()
plt.figure(figsize=(12, 4))
sns.lineplot(data=by_hour, x="DepHour", y="avg_delay", marker="o")
plt.title("Средняя задержка прибытия по часу планового вылета")
plt.ylabel("Средняя задержка, мин")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
by_dow = df.groupBy("DayOfWeek").agg(F.mean("ArrDelay").alias("avg_delay")).toPandas()
plt.figure(figsize=(8, 3.5))
sns.lineplot(data=by_dow, x="DayOfWeek", y="avg_delay", marker="o")
plt.title("Средняя задержка прибытия по дню недели (1 — пн, 7 — вс)")
plt.ylabel("Средняя задержка, мин")
plt.grid(True, alpha=0.3)
plt.show()

Типичная картика авиаотрасли: задержка нарастает в течение дня (рейсы «наследуют» задержки
предыдущих), вечерние вылеты задерживаются сильнее утренних; по дням недели пик — пятница/воскресенье,
пик сезона — лето. Эти закономерности пригодятся как признаки в лабе 2 и как материал выводов.

### Корреляционный анализ количественных признаков

Возьмём сэмпл (5% строк) на драйвер и построим корреляционную матрицу Пирсона средствами pandas —
это стандартный приём: точность оценки на 1,5 млн строк более чем достаточна, а считать
попарные корреляции в Spark дорого.

**На что смотрим:** связь предикторов с целевой `ArrDelay` и мультиколлинеарность между
предикторами (например, `Distance` и `AirTime` почти наверняка коррелируют — это нормально).

In [ ]:
numeric_cols = ["DepDelay", "ArrDelay", "Distance", "AirTime", "TaxiIn", "TaxiOut",
                "CRSElapsedTime", "ActualElapsedTime", "DepHour", "Month", "DayOfWeek"]

sample = df.select(numeric_cols).sample(0.05, seed=42).toPandas()
corr = sample.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, cbar_kws={"shrink": 0.8})
plt.title("Корреляционная матрица количественных признаков (сэмпл 5%)")
plt.tight_layout()
plt.show()

Ожидаемые наблюдения:
* `DepDelay` ↔ `ArrDelay` — сильная положительная связь (самолёт, вылетевший поздно,
  обычно и садится поздно): главное «пороговое» знание для постановки задач ML;
* `Distance` ↔ `AirTime` ↔ `CRSElapsedTime` — почти функциональная связь (дальность определяет
  время полёта) — пример мультиколлинеарности;
* календарные признаки с целевой коррелируют слабо, но по графикам выше видно, что связь есть
  и она нелинейная — линейная корреляция её «не видит», зато её поймают деревья в лабе 2.

### Итоговая схема данных и сохранение обработанной таблицы

Соберём финальный набор: уберём столбцы, дублирующие информацию (`CRSDepTime` заменён на `DepHour`,
полные даты/времена фактические (`DepTime`, `ArrTime`, `ActualElapsedTime`) знать «до вылета» нельзя —
это данные будущего, утечка (leakage); для честной модели «до вылета» их отбрасываем и вернёмся
к этому обсуждению в лабе 2).

Проверим схему, количество строк и сохраним результат в новую Iceberg-таблицу
`lab1_flights_processed_table`.

In [ ]:
df = df.drop("FlightDate", "CRSDepTime", "CRSArrTime", "DepTime", "ArrTime",
             "ActualElapsedTime", "OriginCityName", "DestCityName")
df.printSchema()
print(f"Итоговое количество строк: {df.count():,}")

In [ ]:
df.writeTo("lab1_flights_processed_table").using("iceberg").create()

# При повторном запуске раскомментируйте удаление:
# spark.sql("DROP TABLE IF EXISTS spark_catalog.sidskii_database.lab1_flights_processed_table")

for t in spark.catalog.listTables():
    print(t.name)

### Выводы

1. Датасет из 31,7 млн рейсов (14 ГБ CSV) обработан распределённо на кластере YARN;
   промежуточное и итоговое хранение — таблицы Apache Iceberg (Parquet, в ~30 раз компактнее CSV).
2. Типы признаков определены: количественные (задержки, дальность, время полёта, руление),
   категориальные (авиакомпания, аэропорты), календарные (месяц, день недели, час вылета),
   бинарные статусы (отменён/перенаправлен) — учтены при проектировании обработки.
3. Пропуски: удалены строки отменённых/перенаправленных рейсов (для них не определена цель);
   причины задержек заполнены нулями по семантике BTS; редкие одиночные пропуски удалены.
4. Выбросы: задержки обрезаны до [-60; 300] мин (потеря ~0,x% данных), дальность сохранена —
   экстремальные значения являются реальными дальнемагистральными маршрутами.
5. Инсайты EDA: задержки растут в течение дня; пик — вечер и лето; `DepDelay` сильно коррелирует
   с `ArrDelay`; `Distance`/`AirTime`/`CRSElapsedTime` мультиколлинеарны.
6. Обработанная таблица `lab1_flights_processed_table` готова к обучению моделей в лабе 2.

In [ ]:
spark.stop()